In [15]:
import shutil
import os

if os.path.exists("dataset"):
    shutil.rmtree("dataset")

print("Old dataset removed")

Old dataset removed


In [16]:
import os
import shutil
import random

print("Preparing dataset...")

# Create destination folders
os.makedirs("dataset/train/cats", exist_ok=True)
os.makedirs("dataset/train/dogs", exist_ok=True)
os.makedirs("dataset/test/cats", exist_ok=True)
os.makedirs("dataset/test/dogs", exist_ok=True)

# Split function
def split_data(source, train, test, split=0.8):
    files = os.listdir(source)
    files = [f for f in files if not f.startswith('.')]  # ignore hidden files
    random.shuffle(files)

    split_size = int(len(files) * split)

    for file in files[:split_size]:
        shutil.copy(os.path.join(source, file), train)

    for file in files[split_size:]:
        shutil.copy(os.path.join(source, file), test)

# Use your actual dataset paths
split_data("cats_and_dogs/cats_set", "dataset/train/cats", "dataset/test/cats")
split_data("cats_and_dogs/dogs_set", "dataset/train/dogs", "dataset/test/dogs")

print("Dataset split complete!")

Preparing dataset...
Dataset split complete!


In [17]:
import os
print("Train cats:", len(os.listdir("dataset/train/cats")))
print("Train dogs:", len(os.listdir("dataset/train/dogs")))
print("Test cats:", len(os.listdir("dataset/test/cats")))
print("Test dogs:", len(os.listdir("dataset/test/dogs")))

Train cats: 400
Train dogs: 400
Test cats: 100
Test dogs: 100


In [18]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

print("Loading dataset...")

train_dir = "dataset/train"
test_dir = "dataset/test"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(100, 100),
    batch_size=16
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(100, 100),
    batch_size=16
)

print("Normalizing...")

train_ds = train_ds.map(lambda x, y: (x/255.0, y))
test_ds = test_ds.map(lambda x, y: (x/255.0, y))

print("Building model...")

model = models.Sequential([
    layers.Input(shape=(100,100,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training started...")

history = model.fit(
    train_ds,
    epochs=10,
    validation_data=test_ds
)

print("Evaluating...")

test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

# Sample prediction (fixed warning)
for images, labels in test_ds.take(1):
    preds = model.predict(images)
    print("\nSample Prediction:")
    print("Predicted:", int((preds[0] > 0.5)[0]))
    print("Actual:", labels[0].numpy())

Loading dataset...
Found 800 files belonging to 2 classes.
Found 200 files belonging to 2 classes.
Normalizing...
Building model...
Training started...
Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.5100 - loss: 0.6998 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4900 - loss: 0.6934 - val_accuracy: 0.5050 - val_loss: 0.6927
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5250 - loss: 0.6898 - val_accuracy: 0.4950 - val_loss: 0.6857
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.5850 - loss: 0.6795 - val_accuracy: 0.6500 - val_loss: 0.6539
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6338 - loss: 0.6611 - val_accuracy: 0.6500 - val_loss: 0.6503
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6475 - loss: 0.6443 - val_accuracy: 0.6450 - val_loss: 0.6424
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6913 - loss: 0.6127 -